# StyleTTS 2 Demo (LibriTTS)

Before you run the following cells, please make sure you have downloaded [reference_audio.zip](https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/reference_audio.zip) and unzipped it under the `demo` folder.

### Utils

In [1]:
import torch
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import random
random.seed(0)

import numpy as np
np.random.seed(0)

In [2]:
%cd ..

# scp  user@46.18.108.33:/home/user/voice/StyleTTS2/Models/indic_voices/epoch_2nd_00014.pth /home/cmi_10101/Documents/voice/Hindi/StyleTTS2/Models/indicvoices


/home/user/voice/StyleTTS2


/home/user/anaconda3/envs/styletts/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# load packages
import time
import random
import yaml
import scipy.signal
from munch import Munch
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torchaudio
import librosa
from nltk.tokenize import word_tokenize
import sys 
sys.path.append('/home/user/voice/StyleTTS2')
from models import *
from utils import *
from text_utils import TextCleaner
textclenaer = TextCleaner()

%matplotlib inline

177


In [4]:
to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300)
mean, std = -4, 4

def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask+1, lengths.unsqueeze(1))
    return mask

def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor

def compute_style(path):
    wave, sr = librosa.load(path, sr=24000)
    audio, index = librosa.effects.trim(wave, top_db=30)
    if sr != 24000:
        audio = librosa.resample(audio, sr, 24000)
    mel_tensor = preprocess(audio).to(device)

    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
        ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))

    return torch.cat([ref_s, ref_p], dim=1)

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

### Load models

In [6]:
# load phonemizer
import phonemizer
global_phonemizer = phonemizer.backend.EspeakBackend(language='hi', preserve_punctuation=True,  with_stress=True)

In [7]:
config = yaml.safe_load(open("/home/user/voice/StyleTTS2/Models/indic_voices/config_ft.yml"))

# load pretrained ASR model
ASR_config = config.get('ASR_config', False)
ASR_path = config.get('ASR_path', False)
text_aligner = load_ASR_models(ASR_path, ASR_config)

# load pretrained F0 model
F0_path = config.get('F0_path', False)
pitch_extractor = load_F0_models(F0_path)

# load BERT model
from Utils.PLBERT.util import load_plbert
BERT_path = config.get('PLBERT_dir', False)
plbert = load_plbert(BERT_path)

/home/user/anaconda3/envs/styletts/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/user/anaconda3/envs/styletts/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/user/anaconda3/envs/styletts/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [8]:
model_params = recursive_munch(config['model_params'])
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]

/home/user/anaconda3/envs/styletts/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


/home/user/anaconda3/envs/styletts/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [9]:
params_whole = torch.load("/home/user/voice/StyleTTS2/Models/indic_voices/epoch_2nd_00019.pth", map_location='cpu')
params = params_whole['net']

In [10]:
for key in model:
    if key in params:
        print('%s loaded' % key)
        try:
            model[key].load_state_dict(params[key])
        except:
            from collections import OrderedDict
            state_dict = params[key]
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:] # remove `module.`
                new_state_dict[name] = v
            # load params
            model[key].load_state_dict(new_state_dict, strict=False)
#             except:
#                 _load(params[key], model[key])
_ = [model[key].eval() for key in model]

bert loaded
bert_encoder loaded
predictor loaded
decoder loaded


text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded
text_aligner loaded
pitch_extractor loaded
mpd loaded
msd loaded
wd loaded


In [11]:
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

In [12]:
sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0), # empirical parameters
    clamp=False
)

In [13]:
from phonemizer import phonemize
from phonemizer.separator import Separator

def clean_phonemize(
    text: str,
    language: str = "hi",
    backend: str = "espeak",
    njobs: int = 4
) -> str:
    """
    Phonemize a mixed Hindi-English string into a plain IPA-like string,
    removing any language-switch tags like (en) or (hi).

    Args:
        text (str): Input sentence (e.g., "CRM_SALES AI असिस्टेड कॉल लॉग Received.")
        language (str): Main language code for phonemizer (default 'hi').
        backend (str): Phonemizer backend to use (default 'espeak').
        njobs (int): Number of parallel jobs (default 4).

    Returns:
        str: Phoneme transcription without language-switch flags.
    """
    # 1. Configure separators: no delimiter for phones, a single space for words
    separator = Separator(phone="", word=" ", syllable="")

    # 2. Invoke high-level phonemize with remove-flags policy
    phoneme_list = phonemize(
        [text],
        language=language,
        backend=backend,
        separator=separator,
        strip=True,                   # trim leading/trailing whitespace
        preserve_punctuation=True,    # keep punctuation intact
        njobs=njobs,
        language_switch="remove-flags"  # strip out (en)/(hi) tags :contentReference[oaicite:0]{index=0}
    )

    # 3. Return the first (and only) phoneme string
    return phoneme_list

In [14]:
text = "CRM_SALES AI असिस्टेड कॉल लॉग Received."
cleaned = clean_phonemize(text)
print(cleaned)
print(word_tokenize(cleaned[0]))
print(' '.join(word_tokenize(cleaned[0])))


['siː ɑːɹ ɛm seɪlz eɪ aɪ ʌsɪsʈeːɖ kɔl lɔɡ ɹɪsiːvd.']
['siː', 'ɑːɹ', 'ɛm', 'seɪlz', 'eɪ', 'aɪ', 'ʌsɪsʈeːɖ', 'kɔl', 'lɔɡ', 'ɹɪsiːvd', '.']
siː ɑːɹ ɛm seɪlz eɪ aɪ ʌsɪsʈeːɖ kɔl lɔɡ ɹɪsiːvd .


### Synthesize speech

In [15]:

def inference(text, ref_s, alpha = 0.3, beta = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    # ps = global_phonemizer.phonemize([text])
    ps = clean_phonemize(text)
    # print(ps)
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens.append(0)
    # tokens = [1] + tokens + [2]
    # print(tokens)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 

        s_pred = sampler(noise = torch.randn((1, 256)).unsqueeze(1).to(device), 
                                          embedding=bert_dur,
                                          embedding_scale=embedding_scale,
                                            features=ref_s, # reference from the same speaker as the embedding
                                             num_steps=diffusion_steps).squeeze(1)


        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha)  * ref_s[:, :128]
        s = beta * s + (1 - beta)  * ref_s[:, 128:]

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)
        # if torch.isnan(pred_dur).any():
              #print("!!! NaN found in pred_dur !!!")
              # Decide how to handle
              # return None # Example: Stop processing

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
        wav_out = out.squeeze().cpu().numpy() # weird pulse at the end of the model, need to be fixed later

    # Apply minimal post-processing to avoid muffled sound
    try:
        # Normalize the output (but not too aggressively)
        wav_out = wav_out / (np.max(np.abs(wav_out)) + 1e-7) * 0.9
        
        # Remove DC offset
        wav_out = wav_out - np.mean(wav_out)
        
        # Apply a very gentle high-pass filter just to remove sub-bass rumble
        b, a = scipy.signal.butter(2, 40/(24000/2), 'highpass')
        wav_out = scipy.signal.filtfilt(b, a, wav_out)
    except Exception as e:
        print(f"Warning: Error during audio post-processing: {e}")
    
        
    # return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later
    return wav_out[..., :-50] # weird pulse at the end of the model, need to be fixed later



#### Basic synthesis (5 diffusion steps, seen speakers)

In [16]:
# text = ''' StyleTTS 2 is a text to speech model that leverages style diffusion and adversarial training with large speech language models to achieve human level text to speech synthesis. '''
# text = '''hello sir CRM SALES AI असिस्टेड कॉल लॉग Received.'''
# text = '''जब मैंने सुबह की ठंडी हवा में अपने घर की बालकनी से सूरज की पहली किरणों को धरती पर बिखरते हुए देखा, तो मन एक अनोखी ऊर्जा और शांति से भर गया, मानो प्रकृति ने स्वयं आकर मेरे दिन की सुंदर शुरुआत की हो'''
text = ''' 
नमस्ते Sir / Madam!

मैं Connlee टीम से बोल रही हूँ।

आपन े recently हमारी website visit की थी communication solution के लिए, right?

Connlee एक smart platform है जहा ँ आप *WhatsApp, Voice Call, Screen Sharing* सब कुछ एक ही जगह पर use कर सकते हैं 1 और वो भी with *Al-powered agent* support.

सिर्फ एक app में आपका *customer support, sales 3R collaboration* manage हो सकता है।



'''
# मैं एक छोटा सा * demo schedule* करना चाहती हूँ।

# क्या अभी 2 minute का time मिलेगा आपको ?

In [17]:
texts = {}
texts['Happy'] = "हम आपको अतीत की एक यात्रा पर आमंत्रित करते हुए प्रसन्न हैं, जहाँ हम मानव कृतियों द्वारा निर्मित सबसे अद्भुत स्मारकों का दर्शन करेंगे।"
texts['Sad'] = "हमें यह बताते हुए खेद है कि हमारी समृद्धि और आत्मविश्वास को बहाल करने के प्रयासों में हमें गंभीर असफलता का सामना करना पड़ा है।"
texts['Angry'] = "खगोलशास्त्र का क्षेत्र एक मज़ाक है! इसके सिद्धांत त्रुटिपूर्ण अवलोकनों और पक्षपाती व्याख्याओं पर आधारित हैं!"
texts['Surprised'] = "मुझे विश्वास नहीं हो रहा! क्या आप सचमुच इस तालाब में बैक्टीरिया की एक नई प्रजाति की खोज कर चुके हैं?"


In [18]:
for k, v in texts.items():
	# noise = torch.randn(1,1,256).to(device)
	ref_s = compute_style("/home/user/voice/StyleTTS2/Demo/reference/risha.wav")
	wav = inference(v, ref_s, alpha=0.7, beta=0.7, diffusion_steps=10, embedding_scale=2)
	print(k + ": ")
	import IPython.display as ipd
	display(ipd.Audio(wav, rate=24000, normalize=False))

həm aːpkoː ʌtiːt ki eːk jaːtɾaː pʌɾ aːmʌntɾɪt kʌɾteː hʊeː pɾəsʌnnə hɛ̃ , ɟʌhã həm maːnəʋ kɾɪtɪjõ dʋaːɾaː nɪrmɪt sʌbseː ʌdbʰʊt smaːɾkõ kaː dʌrʃən kəɾẽːɡeː
həm aːpkoː ʌtiːt ki eːk jaːtɾaː pʌɾ aːmʌntɾɪt kʌɾteː hʊeː pɾəsʌnnə hɛ̃ , ɟʌhã həm maːnəʋ kɾɪtɪjõ dʋaːɾaː nɪrmɪt sʌbseː ʌdbʰʊt smaːɾkõ kaː dʌrʃən kəɾẽːɡeː
həm aːpkoː ʌtiːt ki eːk jaːtɾaː pʌɾ aːmʌntɾɪt kʌɾteː hʊeː pɾəsʌnnə hɛ̃ , ɟʌhã həm maːnəʋ kɾɪtɪjõ dʋaːɾaː nɪrmɪt sʌbseː ʌdbʰʊt smaːɾkõ kaː dʌrʃən kəɾẽːɡeː
həm aːpkoː ʌtiːt ki eːk jaːtɾaː pʌɾ aːmʌntɾɪt kʌɾteː hʊeː pɾəsʌnnə hɛ̃ , ɟʌhã həm maːnəʋ kɾɪtɪjõ dʋaːɾaː nɪrmɪt sʌbseː ʌdbʰʊt smaːɾkõ kaː dʌrʃən kəɾẽːɡeː
həm aːpkoː ʌtiːt ki eːk jaːtɾaː pʌɾ aːmʌntɾɪt kʌɾteː hʊeː pɾəsʌnnə hɛ̃ , ɟʌhã həm maːnəʋ kɾɪtɪjõ dʋaːɾaː nɪrmɪt sʌbseː ʌdbʰʊt smaːɾkõ kaː dʌrʃən kəɾẽːɡeː
Happy: 


hʌmẽː jəh bətaːteː hʊeː kʰeːd hɛː kɪ həmaːɾi səmɾɪdʰːɪ ɔːɾ aːtməʋɪʃʋaːs koː bəhaːl kʌɾneː keː pɾəjaːsõ mẽː hʌmẽː ɡəmbʰiːɾ ʌsəpʰəltaː kaː saːmnaː kʌɾnaː pʌr.aː hɛː
hʌmẽː jəh bətaːteː hʊeː kʰeːd hɛː kɪ həmaːɾi səmɾɪdʰːɪ ɔːɾ aːtməʋɪʃʋaːs koː bəhaːl kʌɾneː keː pɾəjaːsõ mẽː hʌmẽː ɡəmbʰiːɾ ʌsəpʰəltaː kaː saːmnaː kʌɾnaː pʌr.aː hɛː
hʌmẽː jəh bətaːteː hʊeː kʰeːd hɛː kɪ həmaːɾi səmɾɪdʰːɪ ɔːɾ aːtməʋɪʃʋaːs koː bəhaːl kʌɾneː keː pɾəjaːsõ mẽː hʌmẽː ɡəmbʰiːɾ ʌsəpʰəltaː kaː saːmnaː kʌɾnaː pʌr.aː hɛː
hʌmẽː jəh bətaːteː hʊeː kʰeːd hɛː kɪ həmaːɾi səmɾɪdʰːɪ ɔːɾ aːtməʋɪʃʋaːs koː bəhaːl kʌɾneː keː pɾəjaːsõ mẽː hʌmẽː ɡəmbʰiːɾ ʌsəpʰəltaː kaː saːmnaː kʌɾnaː pʌr.aː hɛː
Sad: 


kʰəɡoːlʃaːstɾə kaː kʃeːtɾə eːk məzaːk hɛː ! ɪskeː sɪdʰːãt tɾʊʈɪpuːrɳə ʌʋloːknõ ɔːɾ pəkʃəpaːti ʋjaːkʰjaːw pʌɾ aːdʰaːɾɪt hɛ̃ !
kʰəɡoːlʃaːstɾə kaː kʃeːtɾə eːk məzaːk hɛː ! ɪskeː sɪdʰːãt tɾʊʈɪpuːrɳə ʌʋloːknõ ɔːɾ pəkʃəpaːti ʋjaːkʰjaːw pʌɾ aːdʰaːɾɪt hɛ̃ !
kʰəɡoːlʃaːstɾə kaː kʃeːtɾə eːk məzaːk hɛː ! ɪskeː sɪdʰːãt tɾʊʈɪpuːrɳə ʌʋloːknõ ɔːɾ pəkʃəpaːti ʋjaːkʰjaːw pʌɾ aːdʰaːɾɪt hɛ̃ !
Angry: 


mʊɟʰeː ʋɪʃʋaːs nʌhĩ hoː ɾəhaː ! kːjaː aːp sʌcmʊc ɪs taːlaːb mẽː bɛːkʈiɾɪjaː ki eːk nʌi pɾəɟaːtɪ ki kʰoːɟ kʌɾ cʊkeː hɛ̃ ?
mʊɟʰeː ʋɪʃʋaːs nʌhĩ hoː ɾəhaː ! kːjaː aːp sʌcmʊc ɪs taːlaːb mẽː bɛːkʈiɾɪjaː ki eːk nʌi pɾəɟaːtɪ ki kʰoːɟ kʌɾ cʊkeː hɛ̃ ?
mʊɟʰeː ʋɪʃʋaːs nʌhĩ hoː ɾəhaː ! kːjaː aːp sʌcmʊc ɪs taːlaːb mẽː bɛːkʈiɾɪjaː ki eːk nʌi pɾəɟaːtɪ ki kʰoːɟ kʌɾ cʊkeː hɛ̃ ?
Surprised: 


In [16]:
reference_dicts = {}
# reference_dicts['696_92939'] = "Demo/reference_audio/696_92939_000016_000006.wav"
# reference_dicts['1789_142896'] = "Demo/reference_audio/1789_142896_000022_000005.wav"
reference_dicts['Risha'] = "/home/user/voice/StyleTTS2/Demo/reference/risha.wav"

In [17]:
start = time.time()
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    
    wav = inference(text, ref_s, alpha=0.5, beta=0.5, diffusion_steps=10, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized:')
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

RTF = 0.361662
Risha Synthesized:


Reference:


#### With higher diffusion steps (more diverse)

Since the sampler is ancestral, the higher the stpes, the more diverse the samples are, with the cost of slower synthesis speed.

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=10, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized:')
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

#### Basic synthesis (5 diffusion steps, umseen speakers)
The following samples are to reproduce samples in [Section 4](https://styletts2.github.io/#libri) of the demo page. All spsakers are unseen during training. You can compare the generated samples to popular zero-shot TTS models like Vall-E and NaturalSpeech 2.

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['1221-135767'] = ("Demo/reference_audio/1221-135767-0014.wav", "Yea, his honourable worship is within, but he hath a godly minister or two with him, and likewise a leech.")
reference_dicts['5639-40744'] = ("Demo/reference_audio/5639-40744-0020.wav", "Thus did this humane and right minded father comfort his unhappy daughter, and her mother embracing her again, did all she could to soothe her feelings.")
reference_dicts['908-157963'] = ("Demo/reference_audio/908-157963-0027.wav", "And lay me down in my cold bed and leave my shining lot.")
reference_dicts['4077-13754'] = ("Demo/reference_audio/4077-13754-0000.wav", "The army found the people in poverty and left them in comparative wealth.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized: ' + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

### Speech expressiveness

The following section recreates the samples shown in [Section 6](https://styletts2.github.io/#emo) of the demo page. The speaker reference used is `1221-135767-0014.wav`, which is unseen during training. 

#### With `embedding_scale=1`
This is the classifier-free guidance scale. The higher the scale, the more conditional the style is to the input text and hence more emotional.



In [ ]:
ref_s = compute_style("Demo/reference_audio/1221-135767-0014.wav")

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=1)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### With `embedding_scale=2`

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    noise = torch.randn(1,1,256).to(device)
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=2)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### With `embedding_scale=2, alpha = 0.5, beta = 0.9`
`alpha` and `beta` is the factor to determine much we use the style sampled based on the text instead of the reference. The higher the value of `alpha` and `beta`, the more suitable the style it is to the text but less similar to the reference. Using higher beta makes the synthesized speech more emotional, at the cost of lower similarity to the reference. `alpha` determines the timbre of the speaker while `beta` determines the prosody. 

In [ ]:
texts = {}
texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

for k,v in texts.items():
    noise = torch.randn(1,1,256).to(device)
    wav = inference(v, ref_s, diffusion_steps=10, alpha=0.5, beta=0.9, embedding_scale=2)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Zero-shot speaker adaptation
This section recreates the "Acoustic Environment Maintenance" and "Speaker’s Emotion Maintenance" demo in [Section 4](https://styletts2.github.io/#libri) of the demo page. You can compare the generated samples to popular zero-shot TTS models like Vall-E. Note that the model was trained only on LibriTTS, which is about 250 times fewer data compared to those used to trian Vall-E with similar or better effect for these maintainance. 

#### Acoustic Environment Maintenance

Since we want to maintain the acoustic environment in the speaker (timbre), we set  `alpha = 0` to make the speaker as closer to the reference as possible while only changing the prosody according to the text.  

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['3'] = ("Demo/reference_audio/3.wav", "As friends thing I definitely I've got more male friends.")
reference_dicts['4'] = ("Demo/reference_audio/4.wav", "Everything is run by computer but you got to know how to think before you can do a computer.")
reference_dicts['5'] = ("Demo/reference_audio/5.wav", "Then out in LA you guys got a whole another ball game within California to worry about.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.0, beta=0.5, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print('Synthesized: ' + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

#### Speaker’s Emotion Maintenance

Since we want to maintain the emotion in the speaker (prosody), we set  `beta = 0.1` to make the speaker as closer to the reference as possible while having some diversity thruogh the slight timbre change.

In [ ]:
reference_dicts = {}
# format: (path, text)
reference_dicts['Anger'] = ("Demo/reference_audio/anger.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Sleepy'] = ("Demo/reference_audio/sleepy.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Amused'] = ("Demo/reference_audio/amused.wav", "We have to reduce the number of plastic bags.")
reference_dicts['Disgusted'] = ("Demo/reference_audio/disgusted.wav", "We have to reduce the number of plastic bags.")

In [ ]:
noise = torch.randn(1,1,256).to(device)
for k, v in reference_dicts.items():
    path, text = v
    ref_s = compute_style(path)
    start = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.1, diffusion_steps=10, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    import IPython.display as ipd
    print(k + ' Synthesized: ' + text)
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print(k + ' Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))

### Longform Narration

This section includes basic implementation of Algorithm 1 in the paper for consistent longform audio generation. The example passage is taken from [Section 5](https://styletts2.github.io/#long) of the demo page.

In [ ]:
passage = '''If the supply of fruit is greater than the family needs, it may be made a source of income by sending the fresh fruit to the market if there is one near enough, or by preserving, canning, and making jelly for sale. To make such an enterprise a success the fruit and work must be first class. There is magic in the word "Homemade," when the product appeals to the eye and the palate; but many careless and incompetent people have found to their sorrow that this word has not magic enough to float inferior goods on the market. As a rule large canning and preserving establishments are clean and have the best appliances, and they employ chemists and skilled labor. The home product must be very good to compete with the attractive goods that are sent out from such establishments. Yet for first class home made products there is a market in all large cities. All first-class grocers have customers who purchase such goods.'''

In [ ]:
def LFinference(text, s_prev, ref_s, alpha = 0.3, beta = 0.7, t = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    ps = ps.replace('``', '"')
    ps = ps.replace("''", '"')

    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 

        s_pred = sampler(noise = torch.randn((1, 256)).unsqueeze(1).to(device), 
                                          embedding=bert_dur,
                                          embedding_scale=embedding_scale,
                                            features=ref_s, # reference from the same speaker as the embedding
                                             num_steps=diffusion_steps).squeeze(1)
        
        if s_prev is not None:
            # convex combination of previous and current style
            s_pred = t * s_prev + (1 - t) * s_pred
        
        s = s_pred[:, 128:]
        ref = s_pred[:, :128]
        
        ref = alpha * ref + (1 - alpha)  * ref_s[:, :128]
        s = beta * s + (1 - beta)  * ref_s[:, 128:]

        s_pred = torch.cat([ref, s], dim=-1)

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)


        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
        
    return out.squeeze().cpu().numpy()[..., :-100], s_pred # weird pulse at the end of the model, need to be fixed later

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
s_ref = compute_style(path)
sentences = passage.split('.') # simple split by comma
wavs = []
s_prev = None
for text in sentences:
    if text.strip() == "": continue
    text += '.' # add it back
    
    wav, s_prev = LFinference(text, 
                              s_prev, 
                              s_ref, 
                              alpha = 0.3, 
                              beta = 0.9,  # make it more suitable for the text
                              t = 0.7, 
                              diffusion_steps=10, embedding_scale=1.5)
    wavs.append(wav)
print('Synthesized: ')
display(ipd.Audio(np.concatenate(wavs), rate=24000, normalize=False))
print('Reference: ')
display(ipd.Audio(path, rate=24000, normalize=False))

### Style Transfer

The following section demostrates the style transfer capacity for unseen speakers in [Section 6](https://styletts2.github.io/#emo) of the demo page. For this, we set `alpha=0.5, beta = 0.9` for the most pronounced effects (mostly using the sampled style). 

In [ ]:
def STinference(text, ref_s, ref_text, alpha = 0.3, beta = 0.7, diffusion_steps=5, embedding_scale=1):
    text = text.strip()
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)

    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    ref_text = ref_text.strip()
    ps = global_phonemizer.phonemize([ref_text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)

    ref_tokens = textclenaer(ps)
    ref_tokens.insert(0, 0)
    ref_tokens = torch.LongTensor(ref_tokens).to(device).unsqueeze(0)
    
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2) 
        
        ref_input_lengths = torch.LongTensor([ref_tokens.shape[-1]]).to(device)
        ref_text_mask = length_to_mask(ref_input_lengths).to(device)
        ref_bert_dur = model.bert(ref_tokens, attention_mask=(~ref_text_mask).int())
        s_pred = sampler(noise = torch.randn((1, 256)).unsqueeze(1).to(device), 
                                          embedding=bert_dur,
                                          embedding_scale=embedding_scale,
                                            features=ref_s, # reference from the same speaker as the embedding
                                             num_steps=diffusion_steps).squeeze(1)


        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha)  * ref_s[:, :128]
        s = beta * s + (1 - beta)  * ref_s[:, 128:]

        d = model.predictor.text_encoder(d_en, 
                                         s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)


        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, 
                                F0_pred, N_pred, ref.squeeze().unsqueeze(0))
    
        
    return out.squeeze().cpu().numpy()[..., :-50] # weird pulse at the end of the model, need to be fixed later

In [ ]:
# reference texts to sample styles

ref_texts = {}
ref_texts['Happy'] = "We are happy to invite you to join us on a journey to the past, where we will visit the most amazing monuments ever built by human hands."
ref_texts['Sad'] = "I am sorry to say that we have suffered a severe setback in our efforts to restore prosperity and confidence."
ref_texts['Angry'] = "The field of astronomy is a joke! Its theories are based on flawed observations and biased interpretations!"
ref_texts['Surprised'] = "I can't believe it! You mean to tell me that you have discovered a new species of bacteria in this pond?"

In [ ]:
path = "Demo/reference_audio/1221-135767-0014.wav"
s_ref = compute_style(path)

text = "Yea, his honourable worship is within, but he hath a godly minister or two with him, and likewise a leech."
for k,v in ref_texts.items():
    wav = STinference(text, s_ref, v, diffusion_steps=10, alpha=0.5, beta=0.9, embedding_scale=1.5)
    print(k + ": ")
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Speech diversity

This section reproduces samples in [Section 7](https://styletts2.github.io/#var) of the demo page. 

`alpha` and `beta` determine the diversity of the synthesized speech. There are two extreme cases:
- If `alpha = 1` and `beta = 1`, the synthesized speech sounds the most dissimilar to the reference speaker, but it is also the most diverse (each time you synthesize a speech it will be totally different). 
- If `alpha = 0` and `beta = 0`, the synthesized speech sounds the most siimlar to the reference speaker, but it is deterministic (i.e., the sampled style is not used for speech synthesis). 


#### Default setting (`alpha = 0.3, beta=0.7`)
This setting uses 70% of the reference timbre and 30% of the reference prosody and use the diffusion model to sample them based on the text. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.3, beta=0.7, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### Less diverse setting (`alpha = 0.1, beta=0.3`)
This setting uses 90% of the reference timbre and 70% of the reference prosody. This makes it more similar to the reference speaker at cost of less diverse samples. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.1, beta=0.3, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### More diverse setting (`alpha = 0.5, beta=0.95`)
This setting uses 50% of the reference timbre and 5% of the reference prosody (so it uses 100% of the sampled prosody, which makes it more diverse), but this makes it more dissimilar to the reference speaker.  

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0.5, beta=0.95, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### Extreme setting (`alpha = 1, beta=1`)
This setting uses 0% of the reference timbre and prosody and use the diffusion model to sample the entire style. This makes the speaker very dissimilar to the reference speaker. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=1, beta=1, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

#### No variation (`alpha = 0, beta=0`)
This setting uses 0% of the reference timbre and prosody and use the diffusion model to sample the entire style. This makes the speaker very similar to the reference speaker, but there is no variation. 

In [ ]:
# unseen speaker
path = "Demo/reference_audio/1221-135767-0014.wav"
ref_s = compute_style(path)

text = "How much variation is there?"
for _ in range(5):
    wav = inference(text, ref_s, diffusion_steps=10, alpha=0, beta=0, embedding_scale=1)
    display(ipd.Audio(wav, rate=24000, normalize=False))

### Extra fun!

Here we clone some of the authors' voice of the StyleTTS 2 papers with a few seconds of the recording in the wild. None of the voices is in the dataset and all authors agreed to have their voices cloned here.

In [ ]:
text = ''' StyleTTS 2 is a text to speech model that leverages style diffusion and adversarial training with large speech language models to achieve human level text to speech synthesis. '''

In [ ]:
reference_dicts = {}
reference_dicts['Yinghao'] = "Demo/reference_audio/Yinghao.wav"
reference_dicts['Gavin'] = "Demo/reference_audio/Gavin.wav"
reference_dicts['Vinay'] = "Demo/reference_audio/Vinay.wav"
reference_dicts['Nima'] = "Demo/reference_audio/Nima.wav"

In [ ]:
start = time.time()
noise = torch.randn(1,1,256).to(device)
for k, path in reference_dicts.items():
    ref_s = compute_style(path)
    
    wav = inference(text, ref_s, alpha=0.1, beta=0.5, diffusion_steps=5, embedding_scale=1)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print('Speaker: ' + k)
    import IPython.display as ipd
    print('Synthesized:')
    display(ipd.Audio(wav, rate=24000, normalize=False))
    print('Reference:')
    display(ipd.Audio(path, rate=24000, normalize=False))